# Module 1 Practical — Your First LLM Program 🤖

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 1: What are LLMs and how they power ChatGPT, Gemini and Claude**

---

### What you'll do in the next 30 minutes

| # | Experiment | What it proves |
|---|-----------|----------------|
| 1 | Your first API call | You can talk to an LLM from Python, not just a chat window |
| 2 | Next-token prediction | LLMs are "supercharged autocomplete" |
| 3 | Counting tokens | Models see tokens, not words — and you pay per token |
| 4 | Hallucination test | Why we need RAG in Week 2 |
| 5 | Mini retail chatbot | The seed of your Week 1 project (and Week 4 capstone!) |

> 💡 **Remember the big picture:** today's code becomes part of your **Week 1 retail Q&A chatbot**, which grows into your **final capstone** — a deployable AI supply-chain/retail assistant.

## Step 0 — Get your FREE Gemini API key 🔑

We use **Google Gemini's free tier** — no credit card needed.

1. Go to **[https://aistudio.google.com](https://aistudio.google.com)** and sign in with your Google account
2. Click **"Get API key"** → **"Create API key"**
3. Copy the key (it looks like `AIzaSy...`)
4. Paste it in the cell below when asked — **never share it or commit it to GitHub!**

⚠️ The free tier has rate limits (a few requests per minute). If you see a `429` error, just wait a minute and re-run the cell.

In [ ]:
# Install the official Google Gen AI SDK (takes ~30 seconds)
%pip install -q -U google-genai

In [ ]:
# Enter your API key securely (it won't be shown on screen)
from getpass import getpass

API_KEY = getpass("Paste your Gemini API key and press Enter: ")

from google import genai

client = genai.Client(api_key=API_KEY)

# Free-tier friendly model. If this name ever changes, check the
# model list at https://ai.google.dev/gemini-api/docs/models
MODEL = "gemini-2.0-flash"

print("✅ Client ready! Using model:", MODEL)

---
## Experiment 1 — Your first API call 🚀

In the theory session we said: *ChatGPT, Gemini and Claude are just apps — the real power is the model behind them, reachable through an API.*

Let's prove it. Three lines of Python:

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain in 2 sentences what a Large Language Model is, "
             "as if I were a first-year engineering student."
)

print(response.text)

🎉 **You just did what ChatGPT does** — sent text to an LLM and got text back. No chat window needed.

**Try it yourself:** change the prompt above and re-run. Ask it something about your college, your city, or robotics.

In [ ]:
# ✏️ YOUR TURN: write your own prompt below and run this cell
my_prompt = "Write a 4-line poem about robots learning to talk."

response = client.models.generate_content(model=MODEL, contents=my_prompt)
print(response.text)

---
## Experiment 2 — LLMs are "supercharged autocomplete" ⌨️

Theory recap: an LLM's *only* trick is **predicting the next token**, over and over.

Let's watch it complete text the way your phone keyboard does — but smarter:

In [ ]:
prompts = [
    "The capital of France is",
    "Roses are red, violets are",
    "def add(a, b):",           # it even completes CODE the same way!
]

for p in prompts:
    response = client.models.generate_content(
        model=MODEL,
        contents=f"Continue this text with the most natural completion "
                 f"(short, no explanation): {p}"
    )
    print(f"PROMPT: {p!r}")
    print(f"MODEL : {response.text.strip()}")
    print("-" * 60)

**Observation:** whether it's geography, poetry, or Python code — it's all the *same mechanism*: predict what text most plausibly comes next.

This is why one model has so many "superpowers". They all emerge from next-token prediction at massive scale.

---
## Experiment 3 — Counting tokens 🔢

Models don't see words — they see **tokens** (word pieces). This matters because:

- APIs **charge per token** (input + output)
- Models have a **context window** — a maximum number of tokens they can "see"
- More tokens = slower and costlier

Let's count tokens in real sentences:

In [ ]:
sentences = [
    "Hi!",
    "Internship at PMS Robotics!",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious",   # 1 word, many tokens!
    "मुझे रोबोटिक्स बहुत पसंद है",              # Hindi often uses MORE tokens per word
]

for s in sentences:
    result = client.models.count_tokens(model=MODEL, contents=s)
    n_words = len(s.split())
    print(f"{result.total_tokens:>3} tokens | {n_words:>2} words | {s}")

**Notice:**
1. Long rare words split into several tokens
2. Non-English text can cost *more* tokens than English — important in India when building multilingual bots!

Rule of thumb for English: **1 token ≈ ¾ of a word**. We'll use this in Module 4 when we tune cost and context windows.

---
## Experiment 4 — Catching a hallucination 🕵️

Theory recap: LLMs generate **plausible** text, not **verified** text. When they don't know, they often *confidently make things up*. This is called **hallucination**.

Let's trap the model with a question about something that doesn't exist:

In [ ]:
trick_question = (
    "What is the price and battery life of the SmartMart X-500 "
    "wireless barcode scanner? Answer briefly."
)
# (We invented this product — it does not exist!)

response = client.models.generate_content(model=MODEL, contents=trick_question)
print(response.text)

**Did it invent specs for a product we made up?** Or did it admit it doesn't know? Run it a few times — you may get different behaviour.

Now watch what happens when we *give it the facts* (a tiny preview of Week 2's RAG):

In [ ]:
catalog_snippet = """
PRODUCT CATALOG (SmartMart internal):
- SmartMart X-500 wireless barcode scanner: Rs. 4,999, battery 18 hours, 1-year warranty
- SmartMart X-200 wired barcode scanner: Rs. 2,499, no battery, 2-year warranty
"""

grounded_prompt = f"""Answer ONLY using the catalog below.
If the answer is not in the catalog, say "I don't have that information."

{catalog_snippet}

Question: {trick_question}"""

response = client.models.generate_content(model=MODEL, contents=grounded_prompt)
print(response.text)

✨ **Same model, correct answer** — because we grounded it in real data.

In **Week 2** you'll automate this: instead of pasting the catalog by hand, a **vector store (ChromaDB)** will find the right documents for every question. That's RAG.

---
## Experiment 5 — Your first mini retail chatbot 🛒

Time to plant the seed of your **Week 1 project**: a retail Q&A assistant for a fictional store, *SmartMart*.

Two new ideas:
1. **System instruction** — a standing order that shapes the bot's personality and rules (much more on this in Module 2: Prompt Engineering)
2. **Chat session** — keeps the conversation history so the bot remembers what you said earlier

In [ ]:
from google.genai import types

SYSTEM_INSTRUCTION = """You are SmartBot, the friendly assistant of SmartMart retail store in Pune.
Rules:
- Be warm, concise and helpful (2-4 sentences max).
- Store hours: 9 AM - 9 PM, all days. Returns accepted within 7 days with receipt.
- If asked about specific product prices or stock, say you will need to check
  the catalog system (coming in Week 2!).
- Politely refuse questions unrelated to the store."""

chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(system_instruction=SYSTEM_INSTRUCTION),
)

print("✅ SmartBot is ready!")

In [ ]:
# Talk to your bot! (memory works across cells)
reply = chat.send_message("Hi! What time do you close today?")
print("🛒 SmartBot:", reply.text)

In [ ]:
reply = chat.send_message("And can I return a kettle I bought 3 days ago?")
print("🛒 SmartBot:", reply.text)

# Notice it remembers context — try a follow-up like "what if I lost the receipt?"
reply = chat.send_message("What if I lost the receipt?")
print("🛒 SmartBot:", reply.text)

In [ ]:
# 💬 Interactive mode — chat with SmartBot yourself! Type 'quit' to stop.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Bye! See you in Module 2.")
        break
    reply = chat.send_message(user_msg)
    print("🛒 SmartBot:", reply.text)

---
## 🏆 Homework (15 min, counts toward your weekly project)

1. **Personalize SmartBot** — edit the system instruction: give the store your own name, city, hours and 3 rules. Test that it follows them.
2. **Hallucination hunt** — find one more question that makes the raw model hallucinate (Experiment 4 style). Save the prompt + wrong answer; we'll compare in class tomorrow.
3. **Token budget** — use `count_tokens` to measure your system instruction. Can you rewrite it to use fewer tokens *without losing any rule*?

### What's next
**Module 2 — Prompt Engineering:** you'll learn *why* the system instruction worked, and the professional patterns (roles, few-shot examples, output formats) that turn SmartBot from a toy into your Week 1 project.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*